# Articulated Body Dynamics
Reinforcement learning steps environments to collect rollout data. For simulation-based RL, the core is of course about implementing `step()` and running it as fast as possible. Grasping the underlying computation is also critical for identifying physical alignment to reality, analyzing the sensitivity of environment evolution, and knowing the limit/challenging problem instances to simulate. Most of the contents in this module can be found in textbook such as [@featherstone2007] and [@modernrobotics] as standard robot dynamics modelling and computation. In this specific context, dynamics mean the physics law involving force, acceleration and mass/inertia, which are beyond geometry and its differentials in kinematics. For those who are familiar with the Newton's 2nd law, the acceleration of a 1-D particle $a$ is associated to the exerted force $f$ through its mass $m$, as simple as:
$$
\label{eq-newton2ndlaw}
f = \frac{d(mv)}{dt} = m\dot{v} = ma
$$

Things are slightly more complicated for dynamics of a strand of rigid body links connected through articulated joints. Rigid bodies are collections of particles so the force and inertial effects must be accounted together. For this the Newton-Euler equation describes the relation in a form similar to the particle case:

$$
\label{eq-newtoneuler}
\begin{split}
\begin{bmatrix}
\mathbf{\tau}    \\
\mathbf{f}
\end{bmatrix} & = \begin{bmatrix}
\mathcal{I}  &   \mathbf{0}  \\
\mathbf{0}  &   m\mathbf{I}_{3\times 3}
\end{bmatrix}
\begin{bmatrix}
\dot{\mathbf{\omega}}    \\
\dot{\mathbf{v}}
\end{bmatrix} + \begin{bmatrix}
\mathbf{\omega} \times (\mathcal{I}\mathbf{\omega}) \\
m\mathbf{\omega} \times \mathbf{v}
\end{bmatrix}   \\
& = \begin{bmatrix}
\mathcal{I}  &   \mathbf{0}  \\
\mathbf{0}  &   m\mathbf{I}_{3\times 3}
\end{bmatrix}\begin{bmatrix}
\dot{\mathbf{\omega}}    \\
\dot{\mathbf{v}}
\end{bmatrix} - \begin{bmatrix}
\mathbf{\omega}\times  &   \mathbf{0}  \\
\mathbf{v}\times &   \mathbf{\omega}\times
\end{bmatrix}^{\top}
\begin{bmatrix}
\mathcal{I}  &   \mathbf{0}  \\
\mathbf{0}  &   m\mathbf{I}_{3\times 3}
\end{bmatrix}
\begin{bmatrix}
\mathbf{\omega}   \\
\mathbf{v}
\end{bmatrix}
\end{split}
$$
with the skew-asymmetricity of $\mathbf{v}\times$ and $\mathbf{v}\times\mathbf{v}=0$. Here $m$ is the mass of the rigid body and $\mathcal{I}$ is the [inertia tensor](https://en.wikipedia.org/wiki/Moment_of_inertia) about the center-of-mass. $\mathcal{I}$ captures the distribution of material mass of a rigid body and plays the role of particle mass for the rotational motion component. For many basic geometry shapes with a uniform density, the quantity is readily [available](https://en.wikipedia.org/wiki/List_of_moments_of_inertia) from a closed-form integration. [](#eq-newtoneuler) has a resemblance of [](#eq-newton2ndlaw) for also having a linear relation between the mass and acceleration terms. The only structure difference is the extra bias term which is independent of acceleration. This is a fundamental structure for dynamics of rigid and articulated bodies.

:::{note}
The velocity and its differentiation terms here are __spatial velocity and acceleration__ (see the section below). By having the center-of-mass as reference, the acceleration describes the rigid body status as a whole instead of the specific point of the center-of-mass. See section 2.11 in[@featherstone2007] for more detailed discussion about this. 
:::

Rigid bodies in an articulated chain are linked through joints. Applying [](#eq-newtoneuler) to each link in isolation must account for forces due to joints constraining the body motion. In most robotics contexts, the most interested are the resultant motion through the joints due to motors attached to them. The concrete values of contraint forces are usually not part of the desired results. There is thus another way of representing the configuration status of the articulated body and its dynamics with the joint configuration $\mathbf{q}$, called __generalized coordinate__. The dynamic for the generalized coordinate representation is:

$$\label{eq-robodyn}
\mathbf{\tau} = \mathbf{M}(\mathbf{q})\ddot{\mathbf{q}} + \mathbf{C}(\dot{\mathbf{q}}, \mathbf{q}) 
$$

where $\mathbf{\tau}$ is redefined as toruqes applied to the joints, and $\mathbf{M}$ and $\mathbf{C}$ are playing the similar role as the inertia matrix and bias terms. 

A dynamic model as [](#eq-newtoneuler) or [](#eq-robodyn) can be used for following tasks:
* Causal order: given the current state and applied force/torque from a control policy, solve the joint/link acceleration and hence the future state through time-integration. This is called __forward dynamics__ and basically the underlying simulation task of `step()`. 
* Inverse problem: given the current state and the desired joint/link acceleration (and hence the future state), solve the force/torque needed to let the equation hold. This is called __inverse dynamics__ and can play an important role in control.

In what follows, both forward and inverse dynamcis will be covered while a few building blocks are needed before delving into the concrete algorithms. 







